In [ ]:
import os
from getpass import getpass
os.environ["ANTHROPIC_API_KEY"] = getpass("Anthropic API key: ")
os.environ["GEMINI_API_KEY"] = getpass("Gemini API key: ")

In [ ]:
import os, time
import anthropic
from google import genai

DEBUG = True   # extra [debug] printouts; set to False for a quiet full run

def debug(*args):
    if DEBUG:
        print("[debug]", *args)

def http_status(e):
    """HTTP status code of an SDK error, if it has one (Anthropic: status_code, Google: code)."""
    for attr in ("status_code", "code", "status"):
        v = getattr(e, attr, None)
        if isinstance(v, int):
            return v
    return None

# settings (recorded with every call)
CLAUDE_SETTINGS = {
    "provider": "anthropic", "model": "claude-sonnet-5-5", "max_tokens": 2048,
    "effort": "low",                          # lowest effort level
    "thinking": {"type": "between_tools"},    # lowest thinking setting (no up-front thinking without tools)
    "temperature": "provider default (non-default values return HTTP 400)",
    "top_p": "provider default (not settable)", "top_k": "provider default (not settable)",
    "system_prompt": None, "tools": None, "stream": False, "n_messages": 1, "timeout_s": 120,
}
CLAUDE_PRICES = {"input": 2.00, "output": 10.00}

GEMINI_SETTINGS = {
    "provider": "google", "model": "gemini-3.8-flash", "max_tokens": 2048,  # thinking counts toward it
    "thinking_level": "low",                  # lowest supported level ("minimal" is rejected)
    "temperature": "provider default (sampling parameters ignored by the backend)",
    "top_p": "provider default (not settable)", "top_k": "provider default (not settable)",
    "system_prompt": None, "tools": None, "stream": False, "n_messages": 1, "timeout_s": 120,
    "endpoint": "models.generate_content",
}
GEMINI_PRICES = {"input": 0.75, "output": 3.75}

# clients
def make_client(name, factory, env_vars):
    if not any(os.environ.get(v) for v in env_vars):
        print(f"{name}: no API key found (set {' or '.join(env_vars)})")
        return None
    key = next(os.environ[v] for v in env_vars if os.environ.get(v))
    debug(f"{name}: key found, ends in ...{key[-4:]}")
    try:
        return factory()
    except Exception as e:
        print(f"{name}: could not create the client: {type(e).__name__}: {e}")
        return None

claude = make_client("claude", lambda: anthropic.Anthropic(max_retries=0, timeout=120),   # retries are done (and logged) by run_model
                     ["ANTHROPIC_API_KEY"])
gemini = make_client("gemini", lambda: genai.Client(http_options={"timeout": 120_000}),   # milliseconds
                     ["GEMINI_API_KEY", "GOOGLE_API_KEY"])

# one call each
def call_claude(prompt):
    msg = claude.messages.create(
        model=CLAUDE_SETTINGS["model"], max_tokens=CLAUDE_SETTINGS["max_tokens"],
        messages=[{"role": "user", "content": prompt}],
        extra_body={"output_config": {"effort": CLAUDE_SETTINGS["effort"]},
                    "thinking": CLAUDE_SETTINGS["thinking"]})
    u = msg.usage
    return {
        "text": "".join(getattr(b, "text", "") for b in msg.content if getattr(b, "type", None) == "text"),
        "stop_reason": msg.stop_reason, "model_version": msg.model, "response_id": msg.id,
        "usage": {"input_tokens": u.input_tokens,
                  "output_tokens": u.output_tokens,           # includes any thinking tokens
                  "thinking_tokens": None,                    # not reported separately
                  "cache_read_input_tokens": getattr(u, "cache_read_input_tokens", None),
                  "billable_output_tokens": u.output_tokens},
        "raw": msg.model_dump(mode="json"),
    }

def call_gemini(prompt):
    r = gemini.models.generate_content(
        model=GEMINI_SETTINGS["model"], contents=prompt,
        config={"max_output_tokens": GEMINI_SETTINGS["max_tokens"],
                "thinking_config": {"thinking_level": GEMINI_SETTINGS["thinking_level"]}})
    u = r.usage_metadata
    out = (getattr(u, "candidates_token_count", None) or 0) if u else 0
    thoughts = (getattr(u, "thoughts_token_count", None) or 0) if u else 0
    cand = r.candidates[0] if r.candidates else None
    parts = cand.content.parts if cand and cand.content and cand.content.parts else []
    return {
        "text": "".join(p.text for p in parts if getattr(p, "text", None) and not getattr(p, "thought", False)),
        "stop_reason": str(cand.finish_reason) if cand else f"NO_CANDIDATE: {r.prompt_feedback}",
        "model_version": r.model_version, "response_id": r.response_id,
        "usage": {"input_tokens": (getattr(u, "prompt_token_count", None) or 0) if u else 0,
                  "output_tokens": out,                        # visible answer only
                  "thinking_tokens": thoughts,                 # billed at the output rate
                  "cache_read_input_tokens": getattr(u, "cached_content_token_count", None) if u else None,
                  "billable_output_tokens": out + thoughts},
        "raw": r.model_dump(mode="json"),
    }

# key test
HINTS = {400: "a setting was rejected, or no credit left - read the message",
         401: "the key is wrong or revoked", 403: "the key has no access to this model or project",
         404: "model ID not found - check the spelling", 429: "rate limit hit, or no credit/quota left",
         500: "provider error - try again in a minute", 503: "service unavailable - try again later",
         529: "provider overloaded - try again later"}

def test_api(name, client, call_fn, settings):
    print(f"\n--- {name}: {settings['model']} ---")
    if client is None:
        print("SKIPPED: no client (see the message above)")
        return False
    t0 = time.perf_counter()
    try:
        r = call_fn("Reply with the single word OK.")
    except Exception as e:
        s = http_status(e)
        print(f"FAILED: {type(e).__name__} HTTP {s} - {HINTS.get(s, 'see the message below')}")
        print("   ", str(e)[:500])
        return False
    print(f"OK in {time.perf_counter() - t0:.1f}s | served by {r['model_version']} | stop reason {r['stop_reason']}")
    print(f"    reply {r['text']!r} | usage {r['usage']}")
    if r["model_version"] and settings["model"] not in str(r["model_version"]):
        print(f"WARNING: asked for {settings['model']} but was served {r['model_version']}")
    if not r["text"].strip():
        print("WARNING: empty reply - check max_tokens and the thinking setting before the full run")
        return False
    return True

API_OK = {"claude": test_api("claude", claude, call_claude, CLAUDE_SETTINGS),
          "gemini": test_api("gemini", gemini, call_gemini, GEMINI_SETTINGS)}
print("\nAPI_OK =", API_OK)

In [ ]:
import os, sys, stat, json, time, hashlib, platform, datetime, importlib.metadata
import pandas as pd

INPUT_DIR = "question_samples_60"                      # read-only input
PROMPTS_PATH = os.path.join(INPUT_DIR, "final_sample_60_prompts.csv")
OUTPUT_DIRS = {"claude": "claude_responses", "gemini": "gemini_responses"}
N_TRIALS = 11
MAX_ATTEMPTS = 6                         # tries per (prompt, trial); every attempt is logged
NO_RETRY_STATUS = {400, 401, 403, 404}   # the request itself is wrong: retrying won't help
STOP_MESSAGES = ("credit balance is too low",)   # errors that mean "stop the whole session now"
MAX_CONSECUTIVE_FAILURES = 10            # stop the session if this many (prompt, trial) pairs fail in a row
PROGRESS_EVERY = 50                      # print a progress line every N saved responses
EMOTIONS = {"N": "neutral", "F": "fear", "J": "joy"}
EXPECTED = {"claude": ("anthropic", call_claude), "gemini": ("google", call_gemini)}   # guards against mixed-up arguments

# lock the input files: read-only on disk
assert os.path.isdir(INPUT_DIR), f"input folder {INPUT_DIR!r} not found (working dir: {os.getcwd()})"
for f in os.listdir(INPUT_DIR):
    p = os.path.join(INPUT_DIR, f)
    if os.path.isfile(p):
        os.chmod(p, stat.S_IREAD | stat.S_IRGRP | stat.S_IROTH)
        debug("locked read-only:", p)

# load only what is needed (long_answer is never loaded, so it cannot leak into a prompt)
prompts = pd.read_csv(PROMPTS_PATH)[["id", "key", "pub_id", "question", "final_decision", "emotional_question"]]
assert prompts["key"].is_unique, "duplicate keys in the prompt file"
assert prompts["emotional_question"].notna().all(), "some prompts are empty"
assert prompts["key"].str[-2].isin(list(EMOTIONS)).all(), "a key does not end in a known emotion template"
print(f"{len(prompts)} prompts ({prompts['id'].nunique()} questions) x {N_TRIALS} trials "
      f"= {len(prompts) * N_TRIALS} calls per model")
with open(PROMPTS_PATH, "rb") as f:
    PROMPTS_SHA256 = hashlib.sha256(f.read()).hexdigest()
debug(f"{len(prompts)} prompts loaded | sha256 {PROMPTS_SHA256[:12]}... | first key {prompts['key'].iloc[0]}")
debug("prompt lengths (chars): min", prompts["emotional_question"].str.len().min(),
      "max", prompts["emotional_question"].str.len().max())

def now_utc():
    return datetime.datetime.now(datetime.timezone.utc).isoformat()

def paths(model):
    """The model's folder and its two files: outputs (one line per saved response) and log (everything else)."""
    d, in_dir = os.path.abspath(OUTPUT_DIRS[model]), os.path.abspath(INPUT_DIR)
    assert os.path.commonpath([d, in_dir]) != in_dir, f"{d} is inside the input folder"
    os.makedirs(d, exist_ok=True)
    return d, os.path.join(d, f"{model}_outputs.jsonl"), os.path.join(d, f"{model}_log.jsonl")

def ends_with_newline(path):
    if not os.path.exists(path) or os.path.getsize(path) == 0:
        return True
    with open(path, "rb") as f:
        f.seek(-1, os.SEEK_END)
        return f.read(1) == b"\n"

def append_jsonl(path, obj):
    """Append one line; files are never rewritten."""
    line = json.dumps(obj, ensure_ascii=False, default=str) + "\n"
    if not ends_with_newline(path):   # a crash cut the last line short: start a fresh line
        line = "\n" + line
    with open(path, "a", encoding="utf-8") as f:
        f.write(line)

def load_jsonl(path):
    """Read a JSONL file, skipping (and reporting) any broken line, e.g. one cut off by a crash."""
    if not os.path.exists(path):
        return []
    rows, bad = [], 0
    with open(path, encoding="utf-8") as f:
        for n, line in enumerate(f, 1):
            if not line.strip():
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                bad += 1
                print(f"WARNING: {os.path.basename(path)} line {n} is broken and was skipped")
    debug(f"read {len(rows)} lines from {os.path.basename(path)}" + (f" ({bad} broken)" if bad else ""))
    return rows

def sdk_versions():
    out = {}
    for pkg in ["anthropic", "google-genai", "pandas"]:
        try:
            out[pkg] = importlib.metadata.version(pkg)
        except importlib.metadata.PackageNotFoundError:
            out[pkg] = None
    return out

def run_model(model, call_fn, settings, prices):
    """Collect responses only (no analysis). Safe to rerun: it resumes from what is already saved."""
    provider, fn = EXPECTED[model]
    assert settings["provider"] == provider, f"{model}: settings are for {settings['provider']}"
    assert call_fn is fn, f"{model}: wrong call function ({call_fn.__name__})"
    if not globals().get("API_OK", {}).get(model):
        print(f"{model}: skipped - the key test in cell 0 did not pass")
        return
    d, out_path, log_path = paths(model)
    run_id = f"{model}-{datetime.datetime.now(datetime.timezone.utc):%Y%m%dT%H%M%SZ}"

    # every saved response must still match the prompt file (protects against reused ids)
    saved_rows = load_jsonl(out_path)
    current = prompts.set_index("key")
    changed = sorted({r["key"] for r in saved_rows if r["key"] in current.index and (
        int(current.at[r["key"], "pub_id"]) != r["pub_id"]
        or current.at[r["key"], "emotional_question"] != r["prompt"])})
    removed = sorted({r["key"] for r in saved_rows} - set(current.index))
    if changed:
        print(f"STOPPING: {len(changed)} saved keys now point to a different question or prompt "
              f"(e.g. {changed[:5]}). New questions need new ids; existing rows must stay unchanged.")
        return
    if removed:
        print(f"WARNING: {len(removed)} saved keys are no longer in the prompt file (e.g. {removed[:5]}) - kept, not rerun")
    debug(f"{len(saved_rows)} saved responses checked against the prompt file: all match")
    done = {(r["key"], r["trial"]) for r in saved_rows}
    todo = [(r, t) for r in prompts.itertuples() for t in range(1, N_TRIALS + 1) if (r.key, t) not in done]
    print(f"{model}: run {run_id} | {len(done)} saved, {len(todo)} to go -> {d}")
    append_jsonl(log_path, {
        "event": "session_start", "run_id": run_id, "time_utc": now_utc(), "settings": settings,
        "prices_usd_per_mtok": prices, "python": sys.version, "platform": platform.platform(),
        "sdk_versions": sdk_versions(), "n_trials": N_TRIALS, "prompts_file": PROMPTS_PATH,
        "prompts_sha256": PROMPTS_SHA256, "already_saved": len(done), "to_do": len(todo)})

    saved = failed = consecutive_failures = 0
    cost_total, t_session, stop_reason = 0.0, time.perf_counter(), "finished"
    try:
        for r, t in todo:
            result, err = None, None
            for attempt in range(1, MAX_ATTEMPTS + 1):
                started, t0 = now_utc(), time.perf_counter()
                try:
                    result = call_fn(r.emotional_question)
                except Exception as e:
                    err = {"type": type(e).__name__, "http_status": http_status(e), "message": str(e)[:4000]}
                latency = round(time.perf_counter() - t0, 3)
                append_jsonl(log_path, {
                    "event": "attempt", "run_id": run_id, "key": r.key, "trial": t, "attempt": attempt,
                    "time_utc": started, "latency_s": latency, "status": "ok" if result else "error",
                    "error": err, "raw_api_response": result["raw"] if result else None})
                if result:
                    break
                if err["http_status"] in NO_RETRY_STATUS:
                    print(f"{r.key} t{t}: {err['type']} HTTP {err['http_status']} - not retrying: {err['message'][:200]}")
                    break
                wait = min(5 * 2 ** (attempt - 1), 120)
                print(f"{r.key} t{t} attempt {attempt}/{MAX_ATTEMPTS}: {err['type']} HTTP {err['http_status']}, retrying in {wait}s")
                time.sleep(wait)

            if not result:
                if any(s in err["message"].lower() for s in STOP_MESSAGES):
                    stop_reason = "out_of_credit"
                    print(f"STOPPING: out of credit ({err['message'][:200]}). Top up, then rerun this cell to resume.")
                    break
                failed += 1
                consecutive_failures += 1
                print(f"{r.key} t{t}: FAILED after {attempt} attempt(s) - will be retried on the next run")
                if consecutive_failures >= MAX_CONSECUTIVE_FAILURES:
                    stop_reason = "too_many_failures"
                    print(f"STOPPING: {consecutive_failures} failures in a row (outage, quota or bad setting?). "
                          f"Last error: {err['type']} HTTP {err['http_status']}: {err['message'][:300]}")
                    break
                continue
            consecutive_failures = 0

            u = result["usage"]
            cost = ((u.get("input_tokens") or 0) * prices["input"]
                    + (u.get("billable_output_tokens") or 0) * prices["output"]) / 1e6
            cost_total += cost
            append_jsonl(out_path, {
                "run_id": run_id, "provider": settings["provider"], "model_requested": settings["model"],
                "model_version": result["model_version"], "response_id": result["response_id"],
                "key": r.key, "question_id": int(r.id), "pub_id": int(r.pub_id), "question": r.question,
                "ground_truth": str(r.final_decision).lower(), "template_id": r.key[-2:],
                "emotion": EMOTIONS[r.key[-2]], "prompt": r.emotional_question, "trial": t,
                "attempts": attempt, "started_utc": started, "finished_utc": now_utc(), "latency_s": latency,
                "settings": settings, "raw_response": result["text"], "stop_reason": result["stop_reason"],
                **{f"tok_{k}": v for k, v in u.items()}, "cost_usd": round(cost, 8)})
            saved += 1

            if saved <= 3:   # look closely at the first responses
                debug(f"{r.key} t{t} | {latency}s | stop={result['stop_reason']} | usage={u} | ${cost:.6f}")
                debug(f"  raw: {result['text'][:300]!r}")
                if not result["text"].strip():
                    debug("  WARNING: empty response text")
            if saved % PROGRESS_EVERY == 0:
                rate = saved / (time.perf_counter() - t_session)
                print(f"{model}: {saved}/{len(todo)} saved | {failed} failed | ${cost_total:.4f} | "
                      f"~{(len(todo) - saved) / rate / 60:.0f} min left")
    except KeyboardInterrupt:
        stop_reason = "interrupted"
        print(f"\n{model}: interrupted by you. Everything saved so far is kept; rerun the cell to resume.")

    minutes = (time.perf_counter() - t_session) / 60
    append_jsonl(log_path, {
        "event": "session_end", "run_id": run_id, "time_utc": now_utc(), "stop_reason": stop_reason,
        "saved": saved, "failed": failed, "cost_usd": round(cost_total, 6), "minutes": round(minutes, 2),
        "total_saved": len(done) + saved, "total_expected": len(prompts) * N_TRIALS})
    print(f"{model}: session {stop_reason} | {saved} saved this session | {failed} failed | "
          f"${cost_total:.4f} | {minutes:.1f} min | total {len(done) + saved}/{len(prompts) * N_TRIALS}")

In [ ]:
run_model("gemini", call_gemini, GEMINI_SETTINGS, GEMINI_PRICES)

In [ ]:
run_model("claude", call_claude, CLAUDE_SETTINGS, CLAUDE_PRICES)

In [ ]:
# [check] completeness of the saved outputs (read-only)
for model in OUTPUT_DIRS:
    _, out_path, _ = paths(model)
    df = pd.DataFrame(load_jsonl(out_path))
    if df.empty:
        print(f"{model}: no outputs yet")
        continue
    expected = len(prompts) * N_TRIALS
    dupes = df.duplicated(["key", "trial"]).sum()
    missing = expected - df[["key", "trial"]].drop_duplicates().shape[0]
    print(f"{model}: {len(df)} rows | expected {expected} | missing {missing} | duplicates {dupes} | "
          f"versions {sorted(df['model_version'].astype(str).unique())} | cost ${df['cost_usd'].sum():.2f}")

In [ ]:
# [export] readable CSV copies of the outputs (the JSONL files are not touched)
import os
EXPORT_DIR = "readable_exports"
os.makedirs(EXPORT_DIR, exist_ok=True)

for model in OUTPUT_DIRS:
    _, out_path, _ = paths(model)
    rows = load_jsonl(out_path)
    if not rows:
        print(f"{model}: no outputs yet")
        continue
    df = pd.DataFrame(rows).sort_values(["question_id", "template_id", "trial"])
    df = df.drop(columns=["settings", "prompt"])   # long and identical in pattern; still in the JSONL
    csv_path = os.path.join(EXPORT_DIR, f"{model}_outputs.csv")
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")   # utf-8-sig so Excel shows symbols correctly
    print(f"{model}: {len(df)} rows -> {csv_path}")